## Dados Raw, da forma que vieram do site

In [1]:
import pandas as pd

df = pd.read_csv("lncRNA.tsv", sep='\t')
display(df.head())
print(df.shape)

,ncRNA Symbol,ncRNA Category,Species,Disease Name,Sample,Dysfunction Pattern,Validated Method,Description,Clinical Application,Causality,Causal Description,PubMed ID
0,ARHGAP5-AS1,LncRNA,Homo sapiens,"Carcinoma, Hepatocellular",HCC cells and tissues,Epigenetics( m6 A-modified),qRT-PCR,"Among these lncRNAs, we found that ARHGAP5-AS1...",NaN,Yes,ARHGAP5-AS1 remarkably promotes malignant beha...,36354136
1,HOTTIP,LncRNA,Homo sapiens,Osteosarcoma,OS tissues,Interaction(PTBP1/KHSRP ),qRT-PCR//RIP,Our data demonstrated HOTTIP was upregulated i...,NaN,Yes,HOTTIP knockdown resulted in a suppression of ...,33475442
2,HOTTIP,LncRNA,Homo sapiens,Glioma,SF268 cells,Interaction(miRNA-10b/Zeb1/Zeb2),Transfection//Migration Assay//qRT-PCR//MTT//E...,lncRNA HOTTIP was found to be particularly ele...,NaN,Yes,lncRNA HOTTIP was found to be particularly ele...,35402278
3,HOTTIP,LncRNA,Homo sapiens,Retinoblastoma,RB tissues and cells,Interaction(miR-101-3p/STC1 Axis),In Vivo Experiment//RNA Pull-Down//Western Blo...,HOTTIP expression was promoted in RB tissues a...,NaN,Yes,Downregulation of HOTTIP inhibited proliferati...,33784880
4,HOTTIP,LncRNA,Homo sapiens,stomach carcinoma,"cell lines MGC-803, HGC-27, SNU-1, and SGC-7901",regulation[adsorbing miR-615-3p],CCK8//qRT-PCR//Flow Cytometry//Luciferase Repo...,LncRNA HOTTIP was highly expressed in gastric...,NaN,Yes,"After knockdown of lncRNA HOTTIP, the prolife...",32633359


(25440, 12)


In [2]:
print(df.iloc[:, 1].unique()) # tipos de ncRNA
print("\n")
print(df.iloc[:, 2].unique()) # tipos de espécie

<StringArray>
['LncRNA', 'CircRNA', 'miRNA']
Length: 3, dtype: str


<StringArray>
['Homo sapiens', 'Mus musculus', 'Rattus norvegicus', 'Oryctolagus cuniculus']
Length: 4, dtype: str


## Filtrando apenas dados de lncRNA de humanos

In [3]:
df_filtered = df[
    (df.iloc[:, 1] == 'LncRNA') &
    (df.iloc[:, 2] == 'Homo sapiens')
]

print(df_filtered.iloc[:, 1].unique())
print(df_filtered.iloc[:, 2].unique())
print(df_filtered.shape)

<StringArray>
['LncRNA']
Length: 1, dtype: str
<StringArray>
['Homo sapiens']
Length: 1, dtype: str
(12133, 12)


## Removendo colunas desnecessárias

In [4]:
"""
Manter ncRNA Symbol para poder buscar as sequências;
Manter Disease Name para definir os alvos da classficação
"""

columns_to_remove = [
    'ncRNA Category',
    'Species',
    'Sample',
    'Dysfunction Pattern',
    'Validated Method',
    'Description',
    'Clinical Application',
    'Causality',
    'Causal Description',
    'PubMed ID'
]

df_filtered = df_filtered.drop(columns=columns_to_remove)
df_filtered = df_filtered.dropna(subset=['ncRNA Symbol', 'Disease Name'])

display(df_filtered)
print(df_filtered.shape)

,ncRNA Symbol,Disease Name
0,ARHGAP5-AS1,"Carcinoma, Hepatocellular"
1,HOTTIP,Osteosarcoma
2,HOTTIP,Glioma
3,HOTTIP,Retinoblastoma
4,HOTTIP,stomach carcinoma
...,...,...
25434,CASC8,Retinoblastoma
25435,CASC8,Coronary Disease
25436,CASC8,Tuberculosis
25437,CASC8,"Diabetes Mellitus, Type 2"


(12133, 2)


## Criando as Colunas Alvo

In [5]:
freq = df_filtered.iloc[:, 1].value_counts()

values = [10, 15, 20, 25, 30, 35, 40, 50, 60, 70, 80]

for v in values:
    higher_then = freq[freq > v]
    valid = freq[freq > v].index

    df_test = df_filtered[
        df_filtered.iloc[:, 1].isin(valid)
    ]

    print(f"Quantidade de doenças com mais de {v} aparições: {len(higher_then)}")
    print(f"Quantidade de instâncias restante: {df_test.shape[0]}")
    print(f"Porcentagem de instâncias restante: {df_test.shape[0]/df_filtered.shape[0]:.2f}\n")

valid_instances = freq[freq > 50].index
df_filtered = df_filtered[df_filtered.iloc[:, 1].isin(valid_instances)]
print(f"Df após filtragem final: {df_filtered.shape}")

Quantidade de doenças com mais de 10 aparições: 90
Quantidade de instâncias restante: 11301
Porcentagem de instâncias restante: 0.93

Quantidade de doenças com mais de 15 aparições: 72
Quantidade de instâncias restante: 11066
Porcentagem de instâncias restante: 0.91

Quantidade de doenças com mais de 20 aparições: 63
Quantidade de instâncias restante: 10903
Porcentagem de instâncias restante: 0.90

Quantidade de doenças com mais de 25 aparições: 51
Quantidade de instâncias restante: 10628
Porcentagem de instâncias restante: 0.88

Quantidade de doenças com mais de 30 aparições: 46
Quantidade de instâncias restante: 10486
Porcentagem de instâncias restante: 0.86

Quantidade de doenças com mais de 35 aparições: 44
Quantidade de instâncias restante: 10420
Porcentagem de instâncias restante: 0.86

Quantidade de doenças com mais de 40 aparições: 43
Quantidade de instâncias restante: 10381
Porcentagem de instâncias restante: 0.86

Quantidade de doenças com mais de 50 aparições: 41
Quantidade 

### Após análise do grupo, decidimos manter um threshold de >50 aparições; isso implica em 41 rótulos possíveis. Com isso, a rede neural terá instâncias suficientes para o treino, mas ainda pode sofrer com classes minoritárias. Além disso, ainda aplicamos um agrupamento de doenças relacionadas, o que nos permite usar uma abordagem em cascata para treinar a classificação "Macro" e depois a classificação da doença especificamente.

In [6]:
print(df_filtered.iloc[:, 1].unique())

<StringArray>
[               'Carcinoma, Hepatocellular',
                             'Osteosarcoma',
                                   'Glioma',
                           'Retinoblastoma',
                    'Arthritis, Rheumatoid',
 'Squamous Cell Carcinoma of Head and Neck',
                        'Stomach Neoplasms',
                         'Breast Neoplasms',
           'Carcinoma, Non-Small-Cell Lung',
                        'Ovarian Neoplasms',
                 'Leukemia, Myeloid, Acute',
                             'Glioblastoma',
                        'Thyroid Neoplasms',
                 'Nasopharyngeal carcinoma',
                     'Pancreatic Neoplasms',
                     'Colorectal Neoplasms',
                'Thyroid Cancer, Papillary',
                    'Carcinoma, Renal Cell',
                        'Colonic Neoplasms',
                           'Osteoarthritis',
                   'Adenocarcinoma of Lung',
                      'Prostatic Neoplasm

In [7]:
# Dicionário para mapear cada doença para sua categoria
mapping = {
    # Respiratórios e Cabeça/Pescoço
    'Lung Neoplasms': 'Macro_Respiratory_Head',
    'Carcinoma, Non-Small-Cell Lung': 'Macro_Respiratory_Head',
    'Adenocarcinoma of Lung': 'Macro_Respiratory_Head',
    'Squamous Cell Carcinoma of Head and Neck': 'Macro_Respiratory_Head',
    'Nasopharyngeal carcinoma': 'Macro_Respiratory_Head',

    # Gastrointestinais e Anexos
    'Colorectal Neoplasms': 'Macro_Gastrointestinal',
    'Colonic Neoplasms': 'Macro_Gastrointestinal',
    'Stomach Neoplasms': 'Macro_Gastrointestinal',
    'Liver Neoplasms': 'Macro_Gastrointestinal',
    'Carcinoma, Hepatocellular': 'Macro_Gastrointestinal',
    'Cholangiocarcinoma': 'Macro_Gastrointestinal',
    'Pancreatic Neoplasms': 'Macro_Gastrointestinal',
    'Esophageal Neoplasms': 'Macro_Gastrointestinal',
    'Esophageal Squamous Cell Carcinoma': 'Macro_Gastrointestinal',

    # Reprodutivos e Endócrinos
    'Breast Neoplasms': 'Macro_Reproductive_Endocrine',
    'Triple Negative Breast Neoplasms': 'Macro_Reproductive_Endocrine',
    'Ovarian Neoplasms': 'Macro_Reproductive_Endocrine',
    'Endometrial Neoplasms': 'Macro_Reproductive_Endocrine',
    'Uterine Cervical Neoplasms': 'Macro_Reproductive_Endocrine',
    'Prostatic Neoplasms': 'Macro_Reproductive_Endocrine',
    'Thyroid Neoplasms': 'Macro_Reproductive_Endocrine',
    'Thyroid Cancer, Papillary': 'Macro_Reproductive_Endocrine',

    # Nervoso e Ósseo
    'Glioma': 'Macro_Nervous_Skeletal',
    'Glioblastoma': 'Macro_Nervous_Skeletal',
    'Retinoblastoma': 'Macro_Nervous_Skeletal',
    'Osteosarcoma': 'Macro_Nervous_Skeletal',

    # Cardiovascular e Sangue
    'Atherosclerosis': 'Macro_Cardiovascular_Blood',
    'Coronary Artery Disease': 'Macro_Cardiovascular_Blood',
    'Atrial Fibrillation': 'Macro_Cardiovascular_Blood',
    'Leukemia, Myeloid, Acute': 'Macro_Cardiovascular_Blood',
    'Multiple Myeloma': 'Macro_Cardiovascular_Blood',

    # Imunológicas, Infecciosas e Articulares
    'Arthritis, Rheumatoid': 'Macro_Immune_Infectious',
    'Osteoarthritis': 'Macro_Immune_Infectious',
    'Sepsis': 'Macro_Immune_Infectious',
    'COVID-19': 'Macro_Immune_Infectious',

    # Outros (Renais, Pele, etc.)
    'Diabetes Mellitus, Type 1': 'Macro_Others',
    'Depression': 'Macro_Others',
    'Pterygium': 'Macro_Others',
    'Urinary Bladder Neoplasms': 'Macro_Others',
    'Carcinoma, Renal Cell': 'Macro_Others',
    'Melanoma': 'Macro_Others'
}

# Cria a nova coluna de Macrocategorias
#df_filtered = df_filtered.reset_index(drop=True)
df_filtered['Macrocategories'] = df_filtered['Disease Name'].map(mapping)

# Adiciona o prefixo 'Disease_' às doenças específicas (facilita a separação depois)
df_filtered['Disease Name'] = 'Disease_' + df_filtered['Disease Name'].astype(str)

# Cria as matrizes binárias (Encoding)
df_macro = pd.get_dummies(df_filtered['Macrocategories'], dtype=int)
df_micro = pd.get_dummies(df_filtered['Disease Name'], dtype=int)

# Junta os DFs com as colunas alvo a partir do RNA
df_filtered = pd.concat([df_filtered[['ncRNA Symbol']], df_macro, df_micro], axis=1)

# Agrupa por RNA: cada RNA aparece apenas uma vez e todas as 
# doenças relacionadas são mapeadas
df_filtered = df_filtered.groupby('ncRNA Symbol').max().reset_index()
print(df_filtered.shape)
display(df_filtered)
print(df_filtered.info())

(4864, 49)


,ncRNA Symbol,Macro_Cardiovascular_Blood,Macro_Gastrointestinal,Macro_Immune_Infectious,Macro_Nervous_Skeletal,Macro_Others,Macro_Reproductive_Endocrine,Macro_Respiratory_Head,Disease_Adenocarcinoma of Lung,"Disease_Arthritis, Rheumatoid",...,Disease_Pterygium,Disease_Retinoblastoma,Disease_Sepsis,Disease_Squamous Cell Carcinoma of Head and Neck,Disease_Stomach Neoplasms,"Disease_Thyroid Cancer, Papillary",Disease_Thyroid Neoplasms,Disease_Triple Negative Breast Neoplasms,Disease_Urinary Bladder Neoplasms,Disease_Uterine Cervical Neoplasms
0,7SK,0,0,1,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
1,91H,0,1,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,A1BG-AS1,0,1,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,A2M-AS1,0,1,1,0,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
4,AADACL2-AS1,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4859,uc004ags.1,0,0,0,0,0,0,1,1,0,...,0,0,0,0,0,0,0,0,0,0
4860,uc010vaf.1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4861,uc010yty.1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4862,uc021ssa.1,0,0,0,0,0,0,1,1,0,...,0,0,0,0,0,0,0,0,0,0


<class 'pandas.DataFrame'>
RangeIndex: 4864 entries, 0 to 4863
Data columns (total 49 columns):
 #   Column                                            Non-Null Count  Dtype
---  ------                                            --------------  -----
 0   ncRNA Symbol                                      4864 non-null   str  
 1   Macro_Cardiovascular_Blood                        4864 non-null   int64
 2   Macro_Gastrointestinal                            4864 non-null   int64
 3   Macro_Immune_Infectious                           4864 non-null   int64
 4   Macro_Nervous_Skeletal                            4864 non-null   int64
 5   Macro_Others                                      4864 non-null   int64
 6   Macro_Reproductive_Endocrine                      4864 non-null   int64
 7   Macro_Respiratory_Head                            4864 non-null   int64
 8   Disease_Adenocarcinoma of Lung                    4864 non-null   int64
 9   Disease_Arthritis, Rheumatoid                     48

## Agora que as colunas alvo estão devidamente criadas e formatadas, devemos gerar as features. Para isso, vamos mapear cada RNA para sua sequência e aplicar algoritmos de extração de características

In [ ]:
import pandas as pd
from Bio import Entrez
import time
from tqdm import tqdm # Para a barra de progresso

# 1. Configuração Obrigatória do NCBI
# O NCBI exige um email válido. Se você fizer muitas requisições sem email, seu IP é bloqueado.
Entrez.email = "leodemore@gmail.com" 

def ncbi_sequence(simbolo):
    """
    Busca a sequência genética no NCBI a partir de um símbolo de gene/RNA.
    """
    try:
        # Passo 1: Pesquisar o ID associado ao símbolo (filtrando por humano)
        term = f"{simbolo} AND Homo sapiens[Organism] AND ncRNA[Filter]"
        
        handler = Entrez.esearch(db="nucleotide", term=term, retmax=1)
        record = Entrez.read(handler)
        handler.close()
        
        # Se não encontrar nada, retorna None
        if not record["IdList"]:
            return None
            
        id_ncbi = record["IdList"][0]
        
        # Passo 2: Baixar a sequência usando o ID encontrado
        handle_fetch = Entrez.efetch(db="nucleotide", id=id_ncbi, rettype="fasta", retmode="text")
        fasta_data = handle_fetch.read()
        handle_fetch.close()
        
        # Passo 3: Limpar o formato FASTA para pegar só as letras
        lines = fasta_data.split('\n')
        clean_seq = "".join(lines[1:]).replace(' ', '')
        
        return clean_seq
        
    except Exception as e:
        # Se houver erro de conexão ou timeout, ele avisa e continua
        print(f"\nErro ao processar {simbolo}: {e}")
        return None

df_teste = df_filtered.copy()

tqdm.pandas(desc="Baixando Sequências")

# Aplica a função criando uma nova coluna 'Sequencia'
def search(symbol):
    sequence = ncbi_sequence(symbol)
    time.sleep(0.4) # Evita o banimento do IP (limite de 3 requisições/s)
    return sequence

df_teste['Sequence'] = df_teste['ncRNA Symbol'].progress_apply(search)

print(df_teste[['ncRNA Symbol', 'Sequence']])
df_teste.to_csv("df_teste.csv", index=False)

Baixando Sequências:   0%|          | 0/4864 [00:00<?, ?it/s]

Baixando Sequências: 100%|██████████| 4864/4864 [1:28:19<00:00,  1.09s/it]

     ncRNA Symbol                                           Sequence
0             7SK  GGATGTGAGGGCGATCTGGCTGCGACATCTGTCACCCCATTGATCG...
1             91H                                                NaN
2        A1BG-AS1  ATTTTTAGTAGAGACGGGGTTTCGTCATGTTGGGTAGACTGGTCTC...
3         A2M-AS1  CTCCTATGCACCCTCTTTCCATAGGTGATGAGTCACAGGGCTCAGG...
4     AADACL2-AS1  GGTTCCCGCTCGCGCCTCTCCCTCCACACCTCCCTGCAAGCTGAGG...
...           ...                                                ...
4859   uc004ags.1                                                NaN
4860   uc010vaf.1                                                NaN
4861   uc010yty.1                                                NaN
4862   uc021ssa.1                                                NaN
4863   uc061hsf.1                                                NaN

[4864 rows x 2 columns]


## REFAZER LÓGICA DE PROCESSAR OS RNA PARA SUAS SEQUENCIAS GENETICAS POREM TRADUZINDO OS "APELIDOS" PARA O CÓDIGO ESPERADO PELO ICBN